# DATA EXTRACTION VIA API 

In [21]:
import pandas as pd
import yfinance as yf
import wrds
import pandas as pd
import os
import re
import json
from pathlib import Path
from datetime import datetime
import wrds
import psycopg2
from psycopg2.extras import json as psycop_json
import sys

In [22]:
# CONFIQ
WRDS_USERNAME = "tnbtran2023"
INPUT_CSV = "TechCompanyByMarketCap_withCIK.csv"
FILING_TYPES = ["10-K", "10-Q"]
START_DATE = "2010-01-01"
END_DATE = "2014-12-31"

# Output directories
OUTPUT_DIR = "../MDA"
RAW_FILINGS_DIR = "../Filings"

# ========================= CONNECTION SETUP ==================================
def get_wrds_connection(username):
    """
    Establish connection to WRDS PostgreSQL database.
    Follows the exact same connection pattern from the reference notebook.
    You will be prompted for your WRDS password.
    """
    conn = psycopg2.connect(
        host="wrds-pgdata.wharton.upenn.edu",
        database="wrds",
        user=username,
        port=9737
    )
    conn.autocommit = True
    return conn


In [23]:

# STEP 1: READ CSV AND EXTRACT CIK LIST
def load_companies_from_csv(csv_path):
    """
    csv already includes CIK of companies
    """
    df = pd.read_csv(csv_path, dtype=str)
    df.columns = [c.strip() for c in df.columns]

    # Find the CIK column (case-insensitive match)
    cik_col = None
    for col in df.columns:
        if "cik" in col.lower():
            cik_col = col
            break
    if cik_col is None:
        raise ValueError(
            f"No CIK column found in CSV. Columns are: {list(df.columns)}\n"
            f"Please ensure your CSV has a column with 'CIK' in its name."
        )

    # Try to find a company name column (optional, for labelling)
    name_col = None
    for col in df.columns:
        if col.lower() != cik_col.lower() and any(
            kw in col.lower() for kw in ["name", "company", "firm", "ticker"]
        ):
            name_col = col
            break

    companies = []
    for _, row in df.iterrows():
        raw_cik = str(row[cik_col]).strip()
        if not raw_cik or raw_cik.lower() == "nan":
            continue

        # Zero-pad CIK to 10 digits (standard SEC format)
        cik = raw_cik.lstrip("0")
        cik_padded = cik.zfill(10)

        name = ""
        if name_col and pd.notna(row.get(name_col)):
            name = str(row[name_col]).strip()

        companies.append({"cik": cik_padded, "cik_raw": raw_cik, "name": name})

    return companies[0:300] ## should js look at top 300, the rest dont really follow mda standards so the mda extracted are not very good



In [24]:

# =============================================================================
# ================ STEP 2: RETRIEVE FILINGS FROM WRDS ========================
# =============================================================================

def get_filings(conn, cik_list, filing_types, start_date, end_date):
    """
    Retrieve filings from WRDS wrds_sec_search tables.
    Mirrors the reference notebook query pattern exactly.
    
    Parameters:
        conn: psycopg2 connection
        cik_list: list of CIK strings (e.g., ['0001418091'])
        filing_types: list of form types (e.g., ['10-K', '10-Q'])
        start_date: 'YYYY-MM-DD' string
        end_date: 'YYYY-MM-DD' string
    
    Returns list of tuples: (accession, form, filing_date, report_date, 
                              acceptance, registrants, filing_text)
    """
    # Build the SQL IN clauses exactly as the notebook does
    cik_in = ", ".join([f"'{c}'" for c in cik_list])
    form_in = ", ".join([f"'{f}'" for f in filing_types])

    with conn.cursor() as cursor:
        sql_query = f"""
            SELECT
                filing_view.accession,
                filing_view.form,
                filing_view.filing_date,
                filing_view.report_date,
                filing_view.acceptance,
                filing_view.registrants,
                filing_view.filing
            FROM
                wrds_sec_search.filing_view
            JOIN
                wrds_sec_search.registrant 
                ON registrant.accession = filing_view.accession
            WHERE
                form IN ({form_in})
                AND wrds_sec_search.registrant.cik IN ({cik_in})
                AND filing_view.filing_date >= '{start_date}'
                AND filing_view.filing_date <= '{end_date}'
            ORDER BY
                filing_view.filing_date ASC
        """
        cursor.execute(sql_query, [])
        results = cursor.fetchall()
    return results


In [25]:
# =============================================================================
# ================ STEP 3: EXTRACT MD&A SECTION ==============================
# =============================================================================
def flex(word):
    """Allow optional whitespace between every character of a word."""
    return r'\s*'.join(list(word))

mda_end_patterns = re.compile(
    '|'.join([
        rf'item\s*7a[\.\:\s\-\u2014]*{flex("quantitative")}\s+{flex("and")}\s+{flex("qualitative")}',
        rf'item\s*8[\.\:\s\-\u2014]+{flex("financial")}\s+{flex("statements")}',
        rf'item\s*3[\.\:\s\-\u2014]*{flex("quantitative")}\s+{flex("and")}\s+{flex("qualitative")}\s+{flex("disclosure")}s?\s+{flex("about")}',
        rf'item\s*3[\.\:\s\-\u2014]*{flex("qualitative")}\s+{flex("and")}\s+{flex("quantitative")}\s+{flex("disclosure")}s?\s+{flex("about")}',
        rf'item\s*3[\.\:\s\-\u2014]*{flex("quantitative")}\s+{flex("and")}\s+{flex("qualitative")}',
        rf'item\s*3[\.\:\s\-\u2014]*{flex("qualitative")}\s+{flex("and")}\s+{flex("quantitative")}',
        rf'item\s*4[\.\:\s\-\u2014]*{flex("controls")}\s+{flex("and")}\s+{flex("procedures")}',
        rf'(?<!\w){flex("quantitative")}\s+{flex("and")}\s+{flex("qualitative")}\s+{flex("disclosures")}\s+{flex("about")}\s+{flex("market")}\s+{flex("risk")}',
        rf'(?<!\w){flex("quantitative")}\s+{flex("and")}\s+{flex("qualitative")}\s+{flex("disclosures")}\s',
        rf'(?<!\w){flex("controls")}\s+{flex("and")}\s+{flex("procedures")}(?!\s+{flex("or")})',
    ]),
    re.IGNORECASE
)

def extract_mda(text):
    text = re.sub(r'\s+', ' ', text)

    mda_start_patterns = re.compile(
        '|'.join([
            # Standard Item 2 / Item 7 with prefix (most companies)
            r'item\s*2[\.\:\s\-\u2014]*managements?\s+discussion\s+and\s+analysis',
            r'item\s*2[\.\:\s\-\u2014]+management[\'\u2019]?s?\s+discussion\s+and\s+analysis',
            r'item\s*7[\.\:\s\-\u2014]+management[\'\u2019]?s?\s+discussion\s+and\s+analysis',
            r'management[\'\u2019]?s?\s+discussion\s+and\s+analysis\s+of\s+financial\s+condition',
            r'md&a\s+management[\'\u2019]?s?\s+discussion\s+and\s+analysis',
            r'(?<!\w)management[\'\u2019]?s?\s+discussion\s+and\s+analysis(?!\s+of\s+financial)',
        ]),
        re.IGNORECASE
    )
    mda_end_patterns = re.compile(
        '|'.join([
            rf'item\s*7a[\.\:\s\-\u2014]*{flex("quantitative")}\s+{flex("and")}\s+{flex("qualitative")}',
            rf'item\s*8[\.\:\s\-\u2014]+{flex("financial")}\s+{flex("statements")}',
            rf'item\s*3[\.\:\s\-\u2014]*{flex("quantitative")}\s+{flex("and")}\s+{flex("qualitative")}\s+{flex("disclosure")}s?\s+{flex("about")}',
            rf'item\s*3[\.\:\s\-\u2014]*{flex("qualitative")}\s+{flex("and")}\s+{flex("quantitative")}\s+{flex("disclosure")}s?\s+{flex("about")}',
            rf'item\s*3[\.\:\s\-\u2014]*{flex("quantitative")}\s+{flex("and")}\s+{flex("qualitative")}',
            rf'item\s*3[\.\:\s\-\u2014]*{flex("qualitative")}\s+{flex("and")}\s+{flex("quantitative")}',
            rf'item\s*4[\.\:\s\-\u2014]*{flex("controls")}\s+{flex("and")}\s+{flex("procedures")}',
            rf'(?<!\w){flex("quantitative")}\s+{flex("and")}\s+{flex("qualitative")}\s+{flex("disclosures")}\s+{flex("about")}\s+{flex("market")}\s+{flex("risk")}',
            rf'(?<!\w){flex("quantitative")}\s+{flex("and")}\s+{flex("qualitative")}\s+{flex("disclosures")}\s',
            rf'(?<!\w){flex("controls")}\s+{flex("and")}\s+{flex("procedures")}(?!\s+{flex("or")})',
        ]),
        re.IGNORECASE
    )

    start_matches = [m.start() for m in mda_start_patterns.finditer(text)]
    end_matches   = [m.start() for m in mda_end_patterns.finditer(text)]

    if not start_matches or not end_matches:
        return None  # no markers found

    MIN_MDA_LENGTH = 5000

    for start in start_matches:
        end_candidates = [x for x in end_matches if x > start]
        if not end_candidates:
            continue
        end = end_candidates[0]
        if end - start > MIN_MDA_LENGTH:
            return text[start:end]

    return None  # all candidates too short



In [26]:
# =============================================================================
# ================ STEP 4: CLEAN EXTRACTED TEXT ===============================
# =============================================================================

def clean_mda_text(mda_text):
    """
    Clean the extracted MD&A text by removing HTML tags, 
    excessive whitespace, and other artifacts.
    """
    if not mda_text:
        return None

    # Remove HTML/XML tags
    cleaned = re.sub(r'<[^>]+>', ' ', mda_text)

    # Remove XBRL tags
    cleaned = re.sub(r'\{[^}]+\}', '', cleaned)

    # Collapse multiple whitespace/newlines
    cleaned = re.sub(r'\s+', ' ', cleaned)

    # Collapse multiple periods/dashes
    cleaned = re.sub(r'\.{3,}', '...', cleaned)
    cleaned = re.sub(r'-{3,}', '---', cleaned)

    return cleaned.strip()

In [27]:
# ========================================================================
# ================ EXECUTION - combined pipeline =========================
# ========================================================================

def main():

    print("=" * 70)
    print("WRDS SEC MD&A Extractor")
    print("=" * 70)
    print(f"Input CSV:      {INPUT_CSV}")
    print(f"Filing Types:   {FILING_TYPES}")
    print(f"Date Range:     {START_DATE} to {END_DATE}")
    print(f"Raw Filings:    {RAW_FILINGS_DIR}")
    print(f"MD&A Output:    {OUTPUT_DIR}")
    print("=" * 70)

    # ---- Step 1: Load companies from CSV ----
    companies = load_companies_from_csv(INPUT_CSV)
    print(f"\nLoaded {len(companies)} companies from CSV.")
    for c in companies:
        label = c["name"] if c["name"] else f"CIK_{c['cik']}"
        # print(f"  CIK: {c['cik']}  |  {label}")

    # ================================================================
    # PHASE 1 — Fetch from WRDS and save raw filings locally
    # ================================================================
    print(f"\n{'=' * 70}")
    print("PHASE 1: Fetching filings from WRDS and saving locally")
    print(f"{'=' * 70}\n")

    print("Connecting to WRDS...")
    conn = get_wrds_connection(WRDS_USERNAME)
    print("Connected successfully.\n")

    total_companies = len(companies)
    for idx, comp in enumerate(companies, 1):
        cik = comp["cik"]
        label = comp["name"] if comp["name"] else f"CIK_{cik}"
        safe_label = re.sub(r"[^\w\-]", "_", label.strip())
        company_raw_dir = os.path.join(RAW_FILINGS_DIR, safe_label)

        print(f"[{idx}/{total_companies}] {label}  (CIK: {cik})")
        print(f"  Fetching {FILING_TYPES} filings ({START_DATE} to {END_DATE})...")

        cik_variants = list(set([cik, cik.lstrip("0"), cik.zfill(10)]))
        filings = get_filings(conn, cik_variants, FILING_TYPES, START_DATE, END_DATE)
        print(f"  Found {len(filings)} filings.")

        if not filings:
            print(f"  WARNING: No filings found. Skipping.\n")
            continue

        os.makedirs(company_raw_dir, exist_ok=True)
        saved_count = 0
        for filing in filings:
            accession, form, filing_date, report_date, acceptance, registrants, filing_text = filing
            if not filing_text:
                continue
            raw_filename = f"{safe_label}_{form}_{filing_date}.txt"
            raw_filepath = os.path.join(company_raw_dir, raw_filename)
            with open(raw_filepath, "w", encoding="utf-8") as f:
                f.write(filing_text)
            saved_count += 1

        print(f"  Saved {saved_count}/{len(filings)} raw filings to {company_raw_dir}/\n")

    conn.close()
    print("WRDS connection closed.\n")

    # ================================================================
    # PHASE 2 — Filter to CSV companies with local folders, extract MD&A
    # ================================================================
    print(f"{'=' * 70}")
    print("PHASE 2: Extracting MD&A from local raw filings")
    print(f"{'=' * 70}\n")

    # Two-step company selection: name in CSV AND folder exists locally
    summary_rows = []
    total_companies = len(companies)

    for idx, comp in enumerate(companies, 1):
        cik = comp["cik"]
        label = comp["name"] if comp["name"] else f"CIK_{cik}"
        safe_label = re.sub(r"[^\w\-]", "_", label.strip())
        company_raw_dir = os.path.join(RAW_FILINGS_DIR, safe_label)

        # print(f"[{idx}/{total_companies}] {label}  (CIK: {cik})")

        if not os.path.isdir(company_raw_dir):
            print(f"  WARNING: No local folder found at {company_raw_dir}. Skipping.\n")
            summary_rows.append({
                "cik": comp["cik_raw"],
                "company_name": label,
                "filings_found": 0,
                "filings_saved": 0,
                "mda_extracted": 0,
            })
            continue

        raw_files = sorted(f for f in os.listdir(company_raw_dir) if f.endswith(".txt"))
        print(f"  Found {len(raw_files)} local raw filing(s)")

        mda_count = 0
        for raw_file in raw_files:
            # Skip if MDA already extracted
            mda_filename = raw_file.replace(".txt", "_MDA.txt")
            filepath = os.path.join(OUTPUT_DIR, mda_filename)
            if os.path.exists(filepath):
                print(f"    {raw_file} | MD&A already exists. Skipping.")
                continue

            raw_filepath = os.path.join(company_raw_dir, raw_file)
            with open(raw_filepath, "r", encoding="utf-8") as f:
                filing_text = f.read()
       
            # Parse form and date from filename: CompanyName_FORM_DATE.txt
            name_parts = raw_file.replace(".txt", "").split("_")
            filing_date = name_parts[-1]
            form = name_parts[-2]

            mda_raw = extract_mda(filing_text)
            if mda_raw is None:
                print(f"    {form} | {filing_date} | MD&A: NOT FOUND")
                continue

            mda_cleaned = clean_mda_text(mda_raw)
            if mda_cleaned is None or len(mda_cleaned) < 200:
                print(f"    {form} | {filing_date} | MD&A: TOO SHORT")
                continue

            mda_count += 1
            print(f"    {form} | {filing_date} | MD&A: {len(mda_cleaned):,} chars")

            mda_filename = raw_file.replace(".txt", "_MDA.txt")
            filepath = os.path.join(OUTPUT_DIR, mda_filename)
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(mda_cleaned)

        print(f"  Result: {mda_count}/{len(raw_files)} MD&A sections extracted.\n")

        summary_rows.append({
            "cik": comp["cik_raw"],
            "company_name": label,
            "filings_found": len(raw_files),
            "filings_saved": len(raw_files),
            "mda_extracted": mda_count,
        })

    # ---- Save summary ----
    summary_path = os.path.join(OUTPUT_DIR, "_extraction_summary.csv")
    df_summary = pd.DataFrame(summary_rows)
    df_summary.to_csv(summary_path, index=False)

    print(f"{'=' * 70}")
    print("EXTRACTION COMPLETE")
    print(f"{'=' * 70}")
    print(df_summary.to_string(index=False))
    print(f"\nRaw filings:     {os.path.abspath(RAW_FILINGS_DIR)}/")
    print(f"MD&A files:      {os.path.abspath(OUTPUT_DIR)}/")
    print(f"Summary CSV:     {summary_path}")
    print(f"{'=' * 70}")


if __name__ == "__main__":
    main()

WRDS SEC MD&A Extractor
Input CSV:      TechCompanyByMarketCap_withCIK.csv
Filing Types:   ['10-K', '10-Q']
Date Range:     2010-01-01 to 2014-12-31
Raw Filings:    ../Filings
MD&A Output:    ../MDA

Loaded 300 companies from CSV.

PHASE 1: Fetching filings from WRDS and saving locally

Connecting to WRDS...
Connected successfully.

[1/300] NVIDIA  (CIK: 0001045810)
  Fetching ['10-K', '10-Q'] filings (2010-01-01 to 2014-12-31)...
  Found 20 filings.
  Saved 20/20 raw filings to ../Filings/NVIDIA/

[2/300] Apple  (CIK: 0000320193)
  Fetching ['10-K', '10-Q'] filings (2010-01-01 to 2014-12-31)...
  Found 20 filings.
  Saved 20/20 raw filings to ../Filings/Apple/

[3/300] Alphabet (Google)  (CIK: 0001288776)
  Fetching ['10-K', '10-Q'] filings (2010-01-01 to 2014-12-31)...
  Found 20 filings.
  Saved 20/20 raw filings to ../Filings/Alphabet__Google_/

[4/300] Alphabet (Google)  (CIK: 0001652044)
  Fetching ['10-K', '10-Q'] filings (2010-01-01 to 2014-12-31)...
  Found 0 filings.

[5/300]

In [28]:
# check how many docs are available 
from pathlib import Path

def count_folder_items(directory_path):
    path = Path(directory_path)
    return len([*path.iterdir()])
count_folder_items("../MDA")


18184

In [ ]:
# re-extract for problematic file, DO NOT RUN PLS
ITEM3_PATTERN = re.compile(
    r'\bitem\s*3\b',
    re.IGNORECASE
)


def scan_file(filepath: str) -> bool:
    """Return True if the file contains an 'Item 3' match."""
    try:
        with open(filepath, "r", encoding="utf-8", errors="replace") as f:
            for line in f:
                if ITEM3_PATTERN.search(line):
                    return True
    except Exception as e:
        print(f"  [WARNING] Could not read {filepath}: {e}")
    return False

def main():
    if len(sys.argv) < 2:
        print("Usage: python check_item3.py <folder_path>")
        sys.exit(1)

    folder = "../MDA"

    if not os.path.isdir(folder):
        print(f"Error: '{folder}' is not a valid directory.")
        sys.exit(1)

    # Collect all .txt files whose full path contains "10-q" (case-insensitive)
    txt_extensions = {".txt"}
    all_files = []
    for root, _, files in os.walk(folder):
        for fname in files:
            full_path = os.path.join(root, fname)
            if (os.path.splitext(fname)[1].lower() in txt_extensions
                    and "10-q" in full_path.lower()):
                all_files.append(full_path)

    if not all_files:
        print(f"No .txt files found in '{folder}'.")
        sys.exit(0)

    print(f"Scanning {len(all_files)} text file(s) in '{folder}'...\n")

    matches = []
    for fpath in sorted(all_files):
        if scan_file(fpath):
            matches.append(fpath)

    # Report results
    mda_count = 0  # initialize counter

    if matches:
        print(f"Found 'Item 3' (or variation) in {len(matches)} file(s):\n")
        for m in matches:
            print(f"  {m}")
            # Parse form and date from filename: CompanyName_FORM_DATE.txt
            basename = os.path.basename(m)  # get just the filename
            name_parts = basename.replace(".txt", "").split("_")
            filing_date = name_parts[-1]
            form = name_parts[-2]

            # Read the file content
            with open(m, "r", encoding="utf-8", errors="replace") as f:
                filing_text = f.read()

            mda_raw = extract_mda(filing_text)
            if mda_raw is None:
                print(f"    {form} | {filing_date} | MD&A: NOT FOUND")
                continue

            mda_cleaned = clean_mda_text(mda_raw)
            if mda_cleaned is None or len(mda_cleaned) < 200:
                print(f"    {form} | {filing_date} | MD&A: TOO SHORT")
                continue

            mda_count += 1
            print(f"    {form} | {filing_date} | MD&A: {len(mda_cleaned):,} chars")

            mda_filename = basename.replace(".txt", "_MDA.txt")
            filepath = os.path.join(OUTPUT_DIR, mda_filename)
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(mda_cleaned)
            


if __name__ == "__main__":
    main()

Scanning 10732 text file(s) in '../MDA'...

Found 'Item 3' (or variation) in 382 file(s):

  ../MDA/ALT5_Sigma_10-Q_2021-08-16_MDA.txt
  ../MDA/ALT5_Sigma_10-Q_2021-11-15_MDA.txt
  ../MDA/ALT5_Sigma_10-Q_2022-05-12_MDA.txt
  ../MDA/ALT5_Sigma_10-Q_2022-08-15_MDA.txt
  ../MDA/ALT5_Sigma_10-Q_2022-11-14_MDA.txt
  ../MDA/Aehr_Test_Systems_10-Q_2020-01-14_MDA.txt
  ../MDA/Aehr_Test_Systems_10-Q_2020-04-14_MDA.txt
  ../MDA/Aehr_Test_Systems_10-Q_2020-10-14_MDA.txt
  ../MDA/Aehr_Test_Systems_10-Q_2021-01-14_MDA.txt
  ../MDA/Aehr_Test_Systems_10-Q_2021-04-13_MDA.txt
  ../MDA/Agilysys_10-Q_2022-07-29_MDA.txt
  ../MDA/Agilysys_10-Q_2022-10-28_MDA.txt
  ../MDA/Agilysys_10-Q_2023-01-26_MDA.txt
  ../MDA/Agora_io_10-Q_2021-11-10_MDA.txt
  ../MDA/Agora_io_10-Q_2022-05-04_MDA.txt
  ../MDA/Agora_io_10-Q_2022-08-04_MDA.txt
  ../MDA/Agora_io_10-Q_2022-11-03_MDA.txt
  ../MDA/Agora_io_10-Q_2024-05-02_MDA.txt
  ../MDA/Agora_io_10-Q_2024-08-01_MDA.txt
  ../MDA/Agora_io_10-Q_2024-10-31_MDA.txt
  ../MDA/Agora